# 06 — Modo de método: Ejecutar experimentos electroquímicos

El **modo de método** es el flujo de trabajo estándar para ejecutar experimentos electroquímicos completos (LSV, CV, EIS, cronoamperometría, etc.). Carga un archivo de método creado en IviumSoft, opcionalmente modifica sus parámetros desde Python, lo inicia, monitorea el progreso y recupera los datos medidos.

### Descripción del flujo de trabajo

```
load_method()          ← cargar un archivo de método .imf en IviumSoft
    ↓
set_method_parameter() ← (opcional) modificar parámetros programáticamente
    ↓
start_method()         ← iniciar la medición
    ↓
get_available_data_points_number()  ← sondear el progreso en un bucle
    ↓
get_data_point(index)  ← leer datos durante o después de la ejecución
    ↓
save_data()            ← guardar resultados en archivo .idf
```

### Requisitos previos
- IviumSoft instalado, en ejecución, dispositivo conectado
- Un archivo de método (`.imf`) creado en IviumSoft — ver `C:/IviumStat/AppMethods/` para ejemplos integrados
- Referencia de manejo de errores: `01_getting_started.ipynb` — Sección 4

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

## 1. Cargar un archivo de método

`load_method()` carga un archivo de método `.imf` en IviumSoft sin iniciarlo. Esto te permite inspeccionar o modificar parámetros antes de ejecutarlo.

> Actualiza `METHOD_PATH` para que apunte a un archivo de método en tu sistema.

In [ ]:
METHOD_PATH = r"C:/IviumStat/datafiles/TEST1.imf"  # actualizar esta ruta

Pyvium.open_driver()
Pyvium.load_method(METHOD_PATH)
print(f"Loaded: {METHOD_PATH}")

## 2. Modificar parámetros del método

`set_method_parameter(name, value)` cambia parámetros individuales del método cargado. Tanto `name` como `value` son cadenas de texto.

> **Reglas importantes (de la referencia de la DLL de IviumSoft):**
> - Los nombres de parámetros son **sensibles a mayúsculas/minúsculas** y deben coincidir exactamente con la etiqueta del campo visible en la pestaña del método de IviumSoft.
> - Solo se pueden cambiar los **parámetros visibles de nivel superior** — los parámetros ocultos detrás de una casilla de verificación o en un cuadro emergente no son accesibles.
> - Para cambiar el tipo de método, establece `'Method'` primero, luego `'Technique'`. Si los configuras en el orden incorrecto el cambio se ignora silenciosamente.
> - Si la DLL rechaza un parámetro (devuelve código 1 o 2), Pyvium lanza `IllegalCommandError` o `ValueError`. Envuelve las llamadas en `try/except` cuando el nombre o valor del parámetro sea incierto.

### Cambiar el tipo de método

```python
# Siempre establecer Method antes que Technique
Pyvium.set_method_parameter('Method',    'CyclicVoltammetry')
Pyvium.set_method_parameter('Technique', 'Standard')
```

### Parámetros de barrido comunes (los nombres deben coincidir exactamente con la UI de IviumSoft)

| Nombre del parámetro | Método típico | Valor de ejemplo |
|----------------|---------------|---------------|
| `'Method'` | cualquiera | `'CyclicVoltammetry'`, `'LinearSweep'`, `'Potentiostatic'` |
| `'Technique'` | cualquiera | `'Standard'`, `'SampledCurrent'` |
| `'E begin'` | LSV, CV | `'0.0'` |
| `'E end'` | LSV | `'1.0'` |
| `'E step'` | LSV | `'0.005'` |
| `'scanrate'` | CV | `'0.05'` |
| `'duration'` | Cronoamperometría | `'10'` |
| `'freq high'` | EIS | `'100000'` |
| `'freq low'` | EIS | `'0.1'` |

> Los nombres exactos dependen del método cargado. Abre el método en IviumSoft y lee las etiquetas de campo directamente para confirmar la ortografía.

In [ ]:
Pyvium.connect_device() # Necesario para set_method_parameter
# Cambiar tipo de método — siempre Method antes que Technique
Pyvium.set_method_parameter('Method',    'CyclicVoltammetry')
Pyvium.set_method_parameter('Technique', 'Standard')

# Establecer parámetros escalares
Pyvium.set_method_parameter("E begin", "0.0")
Pyvium.set_method_parameter("E end",   "0.8")
Pyvium.set_method_parameter("E step",  "0.005")

# Los parámetros booleanos usan la cadena en minúsculas 'true' o 'false'
Pyvium.set_method_parameter("OCP", "true")      # habilitar pre-medición OCP

# Sub-parámetros: usar notación punto  →  "parent.child"
# Ejemplo: establecer nivel 1 de una cronoimpedanciometría de múltiples niveles
Pyvium.set_method_parameter("I_levels.I[1]", "0.001")   # 1 mA  (siempre en unidades SI)
Pyvium.set_method_parameter("I_levels.I[3]", "-0.001")  # -1 mA

print("Parameters updated")

## 3. Guardar el método modificado

Después de modificar los parámetros puedes guardar el método de vuelta en disco. Esto es útil para construir una biblioteca de plantillas de método parametrizadas desde Python.

In [ ]:
SAVE_METHOD_PATH = r"C:/IviumStat/datafiles/TEST1_modified.imf"

Pyvium.save_method(SAVE_METHOD_PATH)
print(f"Method saved to: {SAVE_METHOD_PATH}")

## 4. Secuencia de pre-medición

Antes de que comiencen los datos de la medición real, un método puede ejecutar hasta cuatro etapas de pre-medición automáticamente:

| Etapa | Qué ocurre |
|-------|------------|
| 1. Purga | Esperar mientras se purga la celda (p.ej., purga de nitrógeno) |
| 2. Pretratamiento | Aplicar un potencial/corriente de acondicionamiento al electrodo |
| 3. Medición OCP | Medir el potencial de circuito abierto y usarlo como referencia |
| 4. Equilibrado | Mantener en el potencial inicial hasta que la celda se estabilice |

> **Importante para el monitoreo de progreso:** `get_available_data_points_number()` devuelve `0` durante todas las etapas de pre-medición aunque el método esté en ejecución activamente. Tu bucle de sondeo estará en `0 puntos` durante toda la duración de estas etapas antes de que aparezca el primer punto de datos real. Tiene esto en cuenta con un tiempo de espera apropiado en lugar de asumir que `0` significa "no iniciado".

> **Botón Continuar:** En la interfaz de usuario manual de IviumSoft, hacer clic en **Continuar** durante la secuencia de pre-medición salta directamente a la medición real (saltando las etapas previas restantes). Al controlar vía Pyvium no hay equivalente Python — las etapas siempre se ejecutan hasta su finalización a menos que todo el método sea cancelado con `abort_method()`.

In [ ]:
# Opción A: iniciar el método actualmente cargado
Pyvium.start_method()
print("Method started")

# Opción B: cargar e iniciar en una llamada
# Pyvium.start_method(r"C:/IviumStat/AppMethods/LSV.imf")

## 5. Monitorear el progreso

`get_available_data_points_number()` devuelve cuántos puntos de datos se han registrado hasta ahora. Sondéalo en un bucle para rastrear el progreso y leer datos en tiempo real.

**Tres advertencias importantes del manual:**

1. **Etapas de pre-medición:** el contador se mantiene en `0` durante Purga, Pretratamiento, OCP y Equilibrado. Agrega un tiempo de espera total para que el bucle no se quede colgado indefinidamente si un método nunca produce datos.

2. **Modo HiSpeed:** IviumSoft entra automáticamente en modo HiSpeed cuando el intervalo de muestreo es < 2 ms. En este modo los datos se almacenan dentro del búfer interno del potenciostato (máx. **8192 puntos**, o 32 768 para CV) y se transfieren al PC solo después de que la medición se completa. El contador se mantiene en `0` todo el tiempo y luego salta al conteo final de golpe. El bucle de sondeo a continuación funciona para ambos casos — solo establece un tiempo de espera total generoso.

3. **MixedMode (umbral dinámico):** Para métodos transitorios (p.ej., un CV con una tasa de barrido muy rápida), IviumSoft puede cambiar dinámicamente entre HiSpeed y transmisión normal a mitad de la medición dependiendo de la tasa real de puntos. Cuando esto ocurre el contador se incrementa parcialmente en pasos en lugar de todo a la vez. La estrategia de sondeo a continuación lo maneja automáticamente.

In [ ]:
TOTAL_EXPECTED_POINTS = 160  # establecer para que coincida con el conteo de puntos esperado de tu método
POLL_INTERVAL_S       = 0.1
OVERALL_TIMEOUT_S     = 300  # cubre etapas de pre-medición + medición real

t_start = time.time()
Pyvium.start_method()
while True:
    n_points = Pyvium.get_available_data_points_number()
    elapsed  = time.time() - t_start
    progress = min(100.0, 100.0 * n_points / TOTAL_EXPECTED_POINTS)
    print(f"\rProgress: {n_points} points ({progress:.0f}%)  [{elapsed:.0f}s]",
          end="", flush=True)

    if n_points >= TOTAL_EXPECTED_POINTS:
        break
    if elapsed > OVERALL_TIMEOUT_S:
        print("\nTimeout — aborting")
        Pyvium.abort_method()
        break
    time.sleep(POLL_INTERVAL_S)

print("\nDone")

## 6. Leer puntos de datos

`get_data_point(index)` devuelve tres valores cuyo significado depende del tipo de método:

| Tipo de método | Valor 1 | Valor 2 | Valor 3 |
|-------------|---------|---------|--------|
| LSV / CV | Potencial (V) | Corriente (A) | 0 |
| Cronoamperometría | Tiempo (s) | Corriente (A) | Potencial (V) |
| EIS | Re(Z) (Ω) | Im(Z) (Ω) | Frecuencia (Hz) |
| Cronopotenciometría | Tiempo (s) | Potencial (V) | Corriente (A) |

El índice es base 1.

In [ ]:
# Leer todos los puntos de datos disponibles
total_points = Pyvium.get_available_data_points_number()

data = []
for i in range(1, total_points + 1):
    v1, v2, v3 = Pyvium.get_data_point(i)
    data.append([v1, v2, v3])

# Construir un DataFrame — los nombres de columnas dependen de tu tipo de método
df = pd.DataFrame(data, columns=["Potential (V)", "Current (A)", "_unused"])
print(f"Read {len(df)} data points")
print(df.head())

## 7. Graficar resultados — Ejemplo LSV

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["Potential (V)"], df["Current (A)"] * 1e6, 'b-', lw=1.5)
ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title("Linear Sweep Voltammetry")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Ejemplo EIS — Diagrama de Nyquist

In [ ]:
# Cargar y ejecutar un método EIS, luego recuperar datos
EIS_METHOD = r"C:/IviumStat/datafiles/EIS.imf" # actualizar esta ruta

# Pyvium.start_method(EIS_METHOD)
# ... esperar finalización ...

total = Pyvium.get_available_data_points_number()
eis_data = []
for i in range(1, total + 1):
    Z_re, Z_im, freq = Pyvium.get_data_point(i)
    eis_data.append([freq, Z_re, Z_im])

eis_df = pd.DataFrame(eis_data, columns=["Frequency (Hz)", "Z_re (Ω)", "Z_im (Ω)"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(eis_df["Z_re (Ω)"], -eis_df["Z_im (Ω)"], 'ro-', markersize=4)
ax.set_xlabel("Z' (Ω)")
ax.set_ylabel("-Z'' (Ω)")
ax.set_title("Nyquist Plot")
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Leer datos de múltiples barridos (CV)

Para voltamperometría cíclica (CV) con múltiples barridos, `get_data_point_from_scan(point_index, scan_index)` permite recuperar datos de cualquier barrido — no solo del último.

Tanto `point_index` como `scan_index` son base 1.

In [ ]:
CV_METHOD       = r"C:/IviumStat/datafiles/CV.imf" # actualizar esta ruta
N_SCANS         = 3
POINTS_PER_SCAN = 200  # actualizar para que coincida con tu método

# Pyvium.start_method(CV_METHOD)
# ... esperar a que terminen todos los barridos ...

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis([i / N_SCANS for i in range(N_SCANS)])

for scan in range(1, N_SCANS + 1):
    potentials, currents = [], []
    for pt in range(1, POINTS_PER_SCAN + 1):
        E, I, _ = Pyvium.get_data_point_from_scan(pt, scan)
        potentials.append(E)
        currents.append(I * 1e6)  # µA
    ax.plot(potentials, currents, color=colors[scan - 1], label=f"Scan {scan}")

ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title(f"Cyclic Voltammetry — {N_SCANS} scans")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Cancelar un método en ejecución

`abort_method()` **no es instantáneo**. Según el manual: la medición se detiene en el siguiente punto de datos. A intervalos de muestreo largos (p.ej., 60 s entre puntos en un barrido EIS lento) la cancelación puede tardar el intervalo restante completo antes de surtir efecto. También puede haber un breve retraso mientras el instrumento descarga datos almacenados en búfer al PC.

> **`IllegalCommandError`:** Si la DLL devuelve el código 1 (no hay medición activa que cancelar), Pyvium lanza `IllegalCommandError`. Al llamar a `abort_method()` de forma defensiva dentro de un bucle de tiempo de espera, captura esta excepción para manejar el caso donde el método ya terminó por su cuenta:
> ```python
> try:
>     Pyvium.abort_method()
> except IllegalCommandError:
>     pass  # nada estaba ejecutándose — eso está bien
> ```

In [ ]:
# Llamar esto mientras un método está en ejecución para detenerlo inmediatamente
# Pyvium.abort_method()
# print("Method aborted")

# Ejemplo: cancelar si la medición tarda demasiado
TIMEOUT_S = 60

# Pyvium.start_method()
t_start = time.time()
while Pyvium.get_available_data_points_number() < TOTAL_EXPECTED_POINTS:
    if time.time() - t_start > TIMEOUT_S:
        Pyvium.abort_method()
        print("Timeout reached — method aborted")
        break
    time.sleep(0.5)
else:
    print("Method completed within timeout")

## 11. Guardar resultados

`save_data()` guarda la última medición en un archivo IDF especificado por el usuario. `save_dataset()` guarda todas las mediciones en la lista de mediciones actual.

**Guardado automático:** IviumSoft guarda automáticamente los datos en su propia ubicación predeterminada cuando termina un método. `save_data()` es un guardado adicional en una ruta que controlas — no reemplaza el guardado automático, y debe llamarse mientras los datos aún están en memoria (antes de que comience el próximo método).

> **Advertencia:** `save_data()` con una ruta inválida **cerrará IviumSoft**. Usa siempre rutas absolutas y verifica que el directorio exista antes de llamarla.

In [ ]:
import os

OUTPUT_DIR  = r"C:/IviumStat/Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "my_measurement.idf")

if os.path.isdir(OUTPUT_DIR):
    Pyvium.save_data(OUTPUT_FILE)
    print(f"Data saved to: {OUTPUT_FILE}")
else:
    print(f"Output directory does not exist: {OUTPUT_DIR}")
    print("Create the directory first or update OUTPUT_DIR")

In [ ]:
# Guardar todas las mediciones en la lista de mediciones en un archivo de conjunto de datos
DATASET_FILE = os.path.join(OUTPUT_DIR, "my_dataset.idf")

if os.path.isdir(OUTPUT_DIR):
    Pyvium.save_dataset(DATASET_FILE)
    print(f"Dataset saved to: {DATASET_FILE}")

## 12. Registro de temperatura (Beta)

En configuraciones multicanal, `update_temperature()` transmite un valor de temperatura a todos los canales. Este es un campo de metadatos — no controla un termostato.

In [ ]:
TEMPERATURE_CELSIUS = 25.0

Pyvium.update_temperature(TEMPERATURE_CELSIUS)
print(f"Temperature set to {TEMPERATURE_CELSIUS} °C")

## 13. Archivo de base de datos SQL

IviumSoft puede almacenar datos de medición en una base de datos SQL. `get_db_file_name()` devuelve la ruta al último archivo de base de datos creado.

In [ ]:
db_path = Pyvium.get_db_file_name()
print(f"SQL database: {db_path}")

## 14. Flujo de trabajo completo del experimento

Juntando todo: una plantilla robusta que maneja errores, monitorea el progreso, lee datos y guarda resultados.

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

METHOD_PATH      = r"C:/IviumStat/datafiles/TEST1.imf"
OUTPUT_PATH      = r"C:/IviumStat/Data/TEST1_result.idf"
EXPECTED_POINTS  = 160
POLL_INTERVAL_S  = 0.2

def run_experiment(method_path, output_path, expected_points):
    print(f"Starting: {os.path.basename(method_path)}")
    Pyvium.start_method(method_path)

    while True:
        n = Pyvium.get_available_data_points_number()
        print(f"\r  {n}/{expected_points} points", end="", flush=True)
        if n >= expected_points:
            break
        time.sleep(POLL_INTERVAL_S)
    print("  — complete")

    rows = []
    for i in range(1, n + 1):
        rows.append(Pyvium.get_data_point(i))
    df = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])

    if os.path.isdir(os.path.dirname(output_path)):
        Pyvium.save_data(output_path)
        print(f"  Saved → {output_path}")

    return df

Pyvium.open_driver()
Pyvium.connect_device()

df = run_experiment(METHOD_PATH, OUTPUT_PATH, EXPECTED_POINTS)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["E (V)"], df["I (A)"] * 1e6, lw=1.5)
ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title("LSV Result")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Pyvium.close_driver()

---

## Resumen

| Tarea | Método |
|------|-------|
| Cargar archivo de método | `load_method(path)` |
| Modificar un parámetro | `set_method_parameter(name, value)` |
| Guardar método modificado | `save_method(path)` |
| Iniciar método (cargado) | `start_method()` |
| Iniciar método (desde archivo) | `start_method(path)` |
| Cancelar método en ejecución | `abort_method()` |
| Verificar progreso | `get_available_data_points_number()` |
| Leer un punto de datos | `get_data_point(index)` → (v1, v2, v3) |
| Leer de un barrido específico | `get_data_point_from_scan(pt, scan)` |
| Guardar último resultado | `save_data(path)` |
| Guardar conjunto de datos completo | `save_dataset(path)` |
| Registrar temperatura | `update_temperature(°C)` |
| Obtener ruta de BD SQL | `get_db_file_name()` |

## Siguiente

- **`07_data_processing.ipynb`** — Parsear archivos IDF guardados sin conexión (no se requiere hardware)
- **`08_batch_and_synchronization.ipynb`** — Coordinar múltiples dispositivos simultáneamente